In [6]:
# If needed, install dependencies (run once in your environment):
# !pip install praat-parselmouth textgrid pandas numpy

import math
import re
from pathlib import Path

import numpy as np
import pandas as pd
import parselmouth
from praatio import tgio

In [7]:
ROOT = Path("/Users/moanason/Downloads/Data_ANA")

# frame definition
FRAME_DUR = 0.020  # 20 ms per frame

TIER_DIAR_A = "Diarisation_A" 
TIER_DIAR_B = "Diarisation_B"
TIER_EVENTS = "TransEvents"

EVENT_MAP = {
    "Turn_A": 10,
    "Turn_B": 11,
    "Silence": 20,
    "Gap": 21,
    "Overlap": 30,
    "Backchannel_A": 31,
    "Backchannel_B": 32,
}

# condition mapping for c column
# c = {0: 'NC1', 1: 'NC2', 2: 'NS'}
COND_CODE_MAP = {
    "NC1": 0,
    "NC2": 1,
    "NS": 2,
}


In [ ]:
def parse_condition_from_name(name: str):
    """
    Parse session number (i) and condition string ("NC1", "NC2", "NS")
    from a filename like 'ref_s01_NC1_processed.TextGrid'
    """
    # pattern: ref_s##_N(C|S)#_processed.TextGrid
    pattern = re.compile(r"ref_s(\d+)_N([CS])(\d*)_processed", re.IGNORECASE)
    m = pattern.search(name)
    if not m:
        raise ValueError(f"Filename does not match expected pattern: {name}")
    
    sess_num = int(m.group(1))   # "01" -> 1
    cs_flag = m.group(2).upper() # "C" or "S"
    rep = m.group(3) or "1"      # repetition number, default "1"

    if cs_flag == "C" and rep == "1":
        cond_str = "NC1"
    elif cs_flag == "C" and rep == "2":
        cond_str = "NC2"
    elif cs_flag == "S":
        cond_str = "NS"
    else:
        raise ValueError(f"Unexpected condition in filename: {name}")
    
    return sess_num, cond_str, cs_flag, rep


def find_mono_audio_paths(sess_num: int, cs_flag: str, rep: str):
    """
    Given session number, C/S flag, and repetition (#),
    return paths to A and B mono audio files:
    p142_s##_A_N(C|S)#_processed.wav
    """
    sess_str = f"{sess_num:02d}"
    cond_part = f"N{cs_flag}{rep}_processed"  # e.g. "NC1_processed"

    fname_A = f"p142_s{sess_str}_A_{cond_part}.wav"
    fname_B = f"p142_s{sess_str}_B_{cond_part}.wav"
    
    path_A = ROOT / fname_A
    path_B = ROOT / fname_B

    if not path_A.is_file():
        raise FileNotFoundError(f"Audio A not found: {path_A}")
    if not path_B.is_file():
        raise FileNotFoundError(f"Audio B not found: {path_B}")
    
    return path_A, path_B


def get_label_at_time(interval_tier: tgio.IntervalTier, time_sec: float) -> str:
    """
    Return the label of the interval containing `time_sec` in a praatio IntervalTier.
    Assumes intervals cover the whole time range (or returns "" if none).
    """
    for start, end, label in interval_tier.entryList:
        if start <= time_sec < end:
            return label.strip()
    return ""


def speech_flag_from_label(label: str) -> int:
    """
    Convert diarisation label to 0/1 speech flag.
    Adjust rules based on your actual labels:
    here: non-empty and not explicitly 'non-speech'/'silence' => speech.
    """
    if not label:
        return 0
    lab = label.strip().lower()
    if lab in {"non-speech", "nonspeech", "silence", ""}:
        return 0
    return 1


def event_code_from_label(label: str) -> int:
    """
    Convert TransEvents label to numeric code e.

    Handles labels:
      - "Turn_A", "Turn_B"
      - "Turn_s01_A", "Turn_s10_B"
      - "Silence", "Pause"
      - "Gap"
      - "Overlap"
      - "Backchannel_A", "Backchannel_B", "Backchannel_s01_A", etc.
    """
    if not label:
        return 20  # treat empty as Silence

    lab = label.strip()

    # normalise for safety!
    low = lab.lower()

    # Silence / pause
    if "silence" in low or "pause" in low:
        return 20

    # Gap
    if "gap" in low:
        return 21

    # Overlap
    if "overlap" in low:
        return 30

    # Backchannels
    if "backchannel" in low:
        if "_a" in low or low.endswith("a"):
            return 31
        if "_b" in low or low.endswith("b"):
            return 32
        # generic backchannel (if you ever have it)
        return 31  # or 32, or 31.5 but int only 😄

    if low.startswith("turn"):
        if "_a" in low or low.endswith("a"):
            return 10
        if "_b" in low or low.endswith("b"):
            return 11

    return EVENT_MAP.get(lab, -1)


def compute_rms(samples: np.ndarray) -> float:
    """
    Compute RMS from a 1D array of samples.
    Returns NaN if empty.
    """
    if samples.size == 0:
        return math.nan
    return float(np.sqrt(np.mean(samples**2)))


In [9]:
all_rows = []

tg_paths = sorted(ROOT.glob("ref_s*_N*_processed.TextGrid"))

print(f"Found {len(tg_paths)} TextGrids.")

for tg_path in tg_paths:
    print(f"Processing TextGrid: {tg_path.name}")

    sess_num, cond_str, cs_flag, rep = parse_condition_from_name(tg_path.name)
    session_id = sess_num
    condition_code = COND_CODE_MAP[cond_str]  # 0/1/2

    path_A, path_B = find_mono_audio_paths(sess_num, cs_flag, rep)

    snd_A = parselmouth.Sound(str(path_A))
    snd_B = parselmouth.Sound(str(path_B))

    sr_A = snd_A.sampling_frequency
    sr_B = snd_B.sampling_frequency
    if sr_A != sr_B:
        raise ValueError(f"Sampling rates differ for session {sess_num}: {sr_A} vs {sr_B}")

    total_dur = snd_A.duration  # seconds

    #  compute PITCH objects for A and B ----
    pitch_A = snd_A.to_pitch(time_step=FRAME_DUR, pitch_floor=75, pitch_ceiling=500)
    pitch_B = snd_B.to_pitch(time_step=FRAME_DUR, pitch_floor=75, pitch_ceiling=500)

    tg = tgio.openTextgrid(str(tg_path))  # <-- no keyword args

    try:
        tier_A = tg.tierDict[TIER_DIAR_A]
        tier_B = tg.tierDict[TIER_DIAR_B]
        tier_events = tg.tierDict[TIER_EVENTS]
    except KeyError as e:
        raise KeyError(
            f"Tier not found in {tg_path.name}: {e}. "
            f"Available tiers: {list(tg.tierDict.keys())}"
        )

    n_frames = int(total_dur / FRAME_DUR) # number of full frames

    # now loop over frames
    for t_idx in range(n_frames):
        t_start = t_idx * FRAME_DUR
        t_end = (t_idx + 1) * FRAME_DUR
        t_center = (t_start + t_end) / 2.0

        lab_A = get_label_at_time(tier_A, t_center)
        lab_B = get_label_at_time(tier_B, t_center)

        s1 = speech_flag_from_label(lab_A)
        s2 = speech_flag_from_label(lab_B)

        lab_event = get_label_at_time(tier_events, t_center)
        e_code = event_code_from_label(lab_event)

        # f0
        f0_A_raw = pitch_A.get_value_at_time(t_center)
        f0_B_raw = pitch_B.get_value_at_time(t_center)

        F0_A = float(f0_A_raw) if (s1 == 1 and not math.isnan(f0_A_raw)) else math.nan
        F0_B = float(f0_B_raw) if (s2 == 1 and not math.isnan(f0_B_raw)) else math.nan

        # rms
        start_sample_A = int(t_start * sr_A)
        end_sample_A = int(t_end * sr_A)
        start_sample_B = int(t_start * sr_B)
        end_sample_B = int(t_end * sr_B)

        samples_A = snd_A.values[0, start_sample_A:end_sample_A]
        samples_B = snd_B.values[0, start_sample_B:end_sample_B]

        rms_A_raw = compute_rms(samples_A)
        rms_B_raw = compute_rms(samples_B)

        RMS_A = float(rms_A_raw) if s1 == 1 else math.nan
        RMS_B = float(rms_B_raw) if s2 == 1 else math.nan

        row = {
            "id": session_id,       # session index
            "cond": condition_code,   # 0=NC1, 1=NC2, 2=NS
            "time": t_idx,            # frame index
            "spk1": s1,              # A speech flag
            "spk2": s2,              # B speech flag
            "event": e_code,           # event code
            "F0_A": F0_A,
            "RMS_A": RMS_A,
            "F0_B": F0_B,
            "RMS_B": RMS_B,
        }
        all_rows.append(row)

print(f"Total frame rows encoded: {len(all_rows)}")


Found 30 TextGrids.
Processing TextGrid: ref_s05_NC1_processed.TextGrid
Processing TextGrid: ref_s05_NC2_processed.TextGrid
Processing TextGrid: ref_s06_NC1_processed.TextGrid
Processing TextGrid: ref_s06_NC2_processed.TextGrid
Processing TextGrid: ref_s07_NC1_processed.TextGrid
Processing TextGrid: ref_s07_NC2_processed.TextGrid
Processing TextGrid: ref_s10_NC1_processed.TextGrid
Processing TextGrid: ref_s10_NC2_processed.TextGrid
Processing TextGrid: ref_s11_NC1_processed.TextGrid
Processing TextGrid: ref_s11_NC2_processed.TextGrid
Processing TextGrid: ref_s12_NC1_processed.TextGrid
Processing TextGrid: ref_s12_NC2_processed.TextGrid
Processing TextGrid: ref_s13_NC1_processed.TextGrid
Processing TextGrid: ref_s13_NC2_processed.TextGrid
Processing TextGrid: ref_s16_NC1_processed.TextGrid
Processing TextGrid: ref_s16_NC2_processed.TextGrid
Processing TextGrid: ref_s17_NC1_processed.TextGrid
Processing TextGrid: ref_s17_NC2_processed.TextGrid
Processing TextGrid: ref_s18_NC1_processed.T

In [10]:
df = pd.DataFrame(all_rows)

# add time in seconds for convenience
df["time_sec"] = df["time"] * FRAME_DUR
# save
out_csv_path = ROOT / "temporal_processed_data_frames.csv"
df.to_csv(out_csv_path, index=False)
print(f"Saved processed data to {out_csv_path}")
df.head(150)

Saved processed data to /Users/moanason/Downloads/Data_ANA/temporal_processed_data_frames.csv


,id,cond,time,spk1,spk2,event,F0_A,RMS_A,F0_B,RMS_B,time_sec
0,5,0,0,0,0,20,NaN,NaN,NaN,NaN,0.00
1,5,0,1,0,0,20,NaN,NaN,NaN,NaN,0.02
2,5,0,2,0,0,20,NaN,NaN,NaN,NaN,0.04
3,5,0,3,0,0,20,NaN,NaN,NaN,NaN,0.06
4,5,0,4,0,0,20,NaN,NaN,NaN,NaN,0.08
...,...,...,...,...,...,...,...,...,...,...,...
145,5,0,145,0,1,11,NaN,NaN,194.910638,0.002918,2.90
146,5,0,146,0,1,11,NaN,NaN,195.975310,0.001172,2.92
147,5,0,147,0,1,11,NaN,NaN,NaN,0.000825,2.94
148,5,0,148,0,1,11,NaN,NaN,NaN,0.000765,2.96
